In [ ]:
import scanpy as sc
import numpy as np
import muon as mu

In [ ]:
adata = sc.read_h5ad("./Data/RNA_ATAC/BMMC/BMMC.h5ad")

adata_gex = adata[:, adata.var['modality'] == "Gene Expression"]
adata_atac = adata[:, adata.var['modality'] == "Peaks"]
print(adata_gex)
print(adata_atac)

View of AnnData object with n_obs × n_vars = 69249 × 13431
    obs: 'cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'DonorID', 'DonorAge', 'DonorGender'
    var: 'gene_id', 'modality'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'
View of AnnData object with n_obs × n_vars = 69249 × 116490
    obs: 'cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'DonorID', 'DonorAge', 'DonorGender'
    var: 'gene_id', 'modality'
    uns: 'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism'
    obsm: 'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
    layers: 'counts'


In [ ]:
sc.pp.normalize_total(adata_gex, target_sum=1e4)
sc.pp.log1p(adata_gex)
sc.pp.highly_variable_genes(adata_gex, n_top_genes=4000, batch_key='batch', subset=True)


sc.pp.normalize_total(adata_atac)
sc.pp.log1p(adata_atac)
sc.pp.highly_variable_genes(adata_atac, n_top_genes=10000, batch_key='batch', flavor="cell_ranger", subset=True)

In [7]:
mdata = mu.MuData({'rna': adata_gex, 'atac': adata_atac})
print(mdata)

MuData object with n_obs × n_vars = 69249 × 14000
  var:	'gene_id', 'modality', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
  2 modalities
    rna:	69249 x 4000
      obs:	'cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'DonorID', 'DonorAge', 'DonorGender'
      var:	'gene_id', 'modality', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
      uns:	'ATAC_gene_activity_var_names', 'dataset_id', 'genome', 'organism', 'log1p', 'hvg'
      obsm:	'ATAC_gene_activity', 'ATAC_lsi_full', 'ATAC_lsi_red', 'ATAC_umap', 'GEX_X_pca', 'GEX_X_umap'
      layers:	'counts'
    atac:	69249 x 10000
      obs:	'cell_type', 'batch', 'GEX_pseudotime_order', 'Samplename', 'Site', 'DonorNumber', 'Modality', 'DonorID', 'DonorAge', 'DonorGender'
      var:	'gene_id', 'modality', 'highly_variable', 'means', 'dispersio

In [ ]:
mu.tl.mofa(mdata, groups_label='batch', gpu_mode=True)


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        
Loaded view='rna' group='s1d1' with N=6224 samples and D=1310 features...
Loaded view='rna' group='s1d2' with N=6740 samples and D=1310 features...
Loaded view='rna' group='s1d3' with N=4279 samples and D=1310 features...
Loaded view='rna' group='s2d1' with N=4220 samples and D=1310 features...
Loaded view='rna' group='s2d4' with N=6111 samples and D=1310 features...
Loaded view

In [ ]:
np.save('MOFA_BMMC.npy', mdata.obsm['X_mofa'])